# PDF 멀티모달 RAG 실습

PDF의 텍스트·표·이미지를 추출하고, 요약을 검색 색인으로 사용한 뒤 원본 자료로 답변합니다. 첫 실행 전 `data/`에 공개 PDF를 넣고 `.env`에 OpenAI API 키를 설정하세요. PDF와 API 키가 없으면 모델 호출 셀은 실행되지 않습니다.

In [2]:
import os
import shutil
import sys
from pathlib import Path

if sys.platform == "win32":
    poppler_root = (
        Path(os.environ.get("LOCALAPPDATA", ""))
        / "Microsoft"
        / "WinGet"
        / "Packages"
        / "oschwartz10612.Poppler_Microsoft.Winget.Source_8wekyb3d8bbwe"
    )
    poppler_bins = sorted(poppler_root.glob("poppler-*/Library/bin"), reverse=True)
    tesseract_dirs = [
        Path(os.environ.get("ProgramFiles", r"C:\Program Files")) / "Tesseract-OCR",
        Path(os.environ.get("ProgramFiles(x86)", r"C:\Program Files (x86)")) / "Tesseract-OCR",
    ]
    native_tool_dirs = [
        *(path for path in poppler_bins if (path / "pdftoppm.exe").is_file()),
        *(path for path in tesseract_dirs if (path / "tesseract.exe").is_file()),
    ]
    os.environ["PATH"] = os.pathsep.join(
        [str(path) for path in native_tool_dirs] + [os.environ.get("PATH", "")]
    )

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — 설치 경로 확인 필요")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — 설치 경로 확인 필요")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.9
가상환경 경로: c:\workspace\multimodal-rag\.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


## 0. 프로젝트 경로와 환경 확인

프로젝트 루트에서 노트북을 실행하거나, `notebooks/` 폴더에서 실행해도 됩니다.

In [3]:
from pathlib import Path
import os
import shutil
import sys
import uuid

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("프로젝트 루트(src 폴더)를 찾을 수 없습니다.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA_DIR = PROJECT_ROOT / "data"
PDF_PATH = DATA_DIR / "your-report.pdf"  # 실제 PDF 파일명으로 바꾸세요.
FIGURES_DIR = DATA_DIR / "figures" / f"{PDF_PATH.stem}-{uuid.uuid4().hex[:8]}"

print("Python:", sys.version.split()[0])
print("환경:", sys.prefix)
print("Poppler:", shutil.which("pdftoppm") or "미설치 / PATH 확인")
print("Tesseract:", shutil.which("tesseract") or "미설치 / PATH 확인")
print("PDF:", PDF_PATH, "존재:", PDF_PATH.is_file())
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"공개 PDF를 data/에 넣고 PDF_PATH를 수정하세요: {PDF_PATH}")

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(".env에 OPENAI_API_KEY를 설정하세요.")

Python: 3.11.9
환경: c:\workspace\multimodal-rag\.venv
Poppler: C:\Users\user\AppData\Local\Microsoft\WinGet\Packages\oschwartz10612.Poppler_Microsoft.Winget.Source_8wekyb3d8bbwe\poppler-25.07.0\Library\bin\pdftoppm.EXE
Tesseract: C:\Program Files\Tesseract-OCR\tesseract.EXE
PDF: C:\workspace\multimodal-rag\data\your-report.pdf 존재: False


FileNotFoundError: 공개 PDF를 data/에 넣고 PDF_PATH를 수정하세요: C:\workspace\multimodal-rag\data\your-report.pdf

## 1. PDF 데이터 추출

`hi_res` 레이아웃 분석을 사용합니다. Poppler와 Tesseract는 Python 패키지가 아닌 OS 프로그램이므로 별도 설치가 필요합니다.

In [ ]:
from multimodal_rag import extract_pdf_elements, split_texts

raw_elements, extracted = extract_pdf_elements(PDF_PATH, FIGURES_DIR)
text_chunks = split_texts(extracted.texts, chunk_size=2000, chunk_overlap=200)
texts = extracted.texts
tables = extracted.tables
image_paths = extracted.image_paths

print(f"전체 추출 요소: {len(raw_elements)}")
print(f"텍스트 블록: {len(texts)} / 분할 텍스트: {len(text_chunks)}")
print(f"표: {len(tables)} / 이미지: {len(image_paths)}")
print("첫 텍스트:", text_chunks[0][:500] if text_chunks else "텍스트 없음")
print("추출 이미지:", image_paths[:5])
if not text_chunks and not tables and not image_paths:
    raise RuntimeError("PDF에서 텍스트, 표, 이미지를 추출하지 못했습니다.")

## 2. 텍스트와 표 요약

요약은 벡터 검색용 색인입니다. 답변 생성 시에는 검색된 원본 텍스트·표를 사용합니다.

In [ ]:
from multimodal_rag import generate_text_summaries

text_summaries, table_summaries = generate_text_summaries(
    text_chunks,
    tables,
    summarize_texts=True,
    model_name="gpt-4o-mini",
    max_concurrency=3,
)
print("텍스트 요약 수:", len(text_summaries))
print("표 요약 수:", len(table_summaries))
print("첫 텍스트 요약:", text_summaries[0] if text_summaries else "없음")
print("첫 표 요약:", table_summaries[0] if table_summaries else "표 객체 없음")

## 3. 이미지 요약

이미지 원본은 data URI로 보관하고 한국어 설명을 임베딩 검색에 사용합니다. 비용을 관리하려면 PDF에서 추출한 이미지 파일을 먼저 직접 확인하세요.

In [ ]:
from multimodal_rag import generate_image_summaries

image_data_uris, image_summaries = generate_image_summaries(
    image_paths,
    model_name="gpt-4o-mini",
)
print("이미지 요약 수:", len(image_summaries))
print("첫 이미지 요약:", image_summaries[0] if image_summaries else "이미지 없음")
if image_data_uris:
    print("첫 이미지 data URI 접두사:", image_data_uris[0][:40])

## 4. 다중 벡터 검색기 구성

요약은 Chroma 벡터 저장소에, 원본은 같은 ID로 연결된 메모리 저장소에 둡니다. 노트북을 재시작하면 메모리 저장소를 다시 채워야 하므로 추출·요약·색인 셀을 순서대로 실행하세요.

In [ ]:
from multimodal_rag import add_documents, create_retriever

retriever = create_retriever(collection_name=f"multimodal_rag_{uuid.uuid4().hex}")
indexed_counts = {}
indexed_counts["text"] = len(add_documents(retriever, text_summaries, text_chunks, source_type="text"))
indexed_counts["table"] = len(add_documents(retriever, table_summaries, tables, source_type="table"))
indexed_counts["image"] = len(add_documents(retriever, image_summaries, image_data_uris, source_type="image"))
print("색인 완료:", indexed_counts)

## 5. 텍스트 RAG와 멀티모달 RAG 비교

아래 질문은 선택한 PDF의 그래프/표를 실제로 묻도록 고치세요. 같은 질문으로 텍스트 요약 검색 결과만 사용하는 기준선과 원본 이미지·표까지 전달하는 체인을 비교합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from multimodal_rag import build_multimodal_chain

multimodal_chain = build_multimodal_chain(retriever)
text_only_prompt = ChatPromptTemplate.from_template(
    "다음 텍스트 검색 결과만 근거로 한국어로 답하세요. 모르면 모른다고 답하세요.\n"
    "질문: {question}\n검색 요약:\n{context}"
)
text_only_chain = text_only_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

query = "문서의 그래프 또는 표에서 확인되는 핵심 추세와 근거 수치를 설명해줘."
summary_hits = retriever.vectorstore.similarity_search(query, k=4)
baseline_answer = text_only_chain.invoke({
    "question": query,
    "context": "\n\n".join(doc.page_content for doc in summary_hits),
})
multimodal_answer = multimodal_chain.invoke(query)
print("[텍스트 요약만 사용]\n", baseline_answer)
print("\n[요약으로 검색 후 원본으로 답변]\n", multimodal_answer)

## 6. 질의응답 5건 및 근거 기록

질문을 자료에 맞게 수정한 뒤 실행합니다. 아래 출력 결과와 근거 종류를 검토하고 README의 제출 표에 실제 답변을 옮기세요. 모델 답변을 검증하지 않은 채 제출하지 마세요.

In [ ]:
questions = [
    "자료에서 가장 큰 변화가 나타난 기간과 수치는 무엇인가요?",
    "그래프의 전반적인 추세를 근거 수치와 함께 설명해 주세요.",
    "표에서 비교하는 항목 중 가장 높거나 낮은 값은 무엇인가요?",
    "자료의 두 기간 또는 두 항목 사이에 어떤 차이가 있나요?",
    "그래프나 표가 뒷받침하는 핵심 결론은 무엇인가요?",
]

for number, question in enumerate(questions, start=1):
    retrieved_sources = retriever.vectorstore.similarity_search_with_relevance_scores(question, k=4)
    source_types = sorted({doc.metadata.get("source_type", "unknown") for doc, _ in retrieved_sources})
    answer = multimodal_chain.invoke(question)
    print(f"\n### {number}. {question}")
    print("근거 종류:", ", ".join(source_types) if source_types else "검색 결과 없음")
    print("답변:", answer)

## 7. 선택: Ollama 로컬 번역 체인

로컬 Ollama 서버를 설치하고 `ollama pull llama3.1:8b`를 실행한 경우에만 사용합니다. 이 셀은 선택 사항이며 기본 답변은 이미 한국어로 생성됩니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM

translation_prompt = ChatPromptTemplate.from_template(
    "다음 답변을 자연스러운 한국어로 번역하세요. 숫자, 단위, 고유명사와 사실관계는 유지하세요.\n\n{answer}"
)
translator = OllamaLLM(model="llama3.1:8b")
translation_chain = translation_prompt | translator | StrOutputParser()
translated_answer = translation_chain.invoke({"answer": multimodal_answer})
print(translated_answer)

## 8. 결과 해석 및 체크리스트

- 요약으로 검색하고 원본으로 답변하는 이유: 요약은 검색 효율을 높이지만 세부 수치/레이아웃이 손실될 수 있으므로, 답변 단계에서 원본을 다시 전달합니다.
- 표 요약이 비어 있으면 실제 PDF에 표 객체가 있는지, 추출 결과가 `hi_res`인지 확인합니다. 이미지로 삽입된 표는 이미지 근거로 처리될 수 있습니다.
- PDF와 추출 이미지는 과제 제출 저장소에 포함하지 말고, OpenAI API 키도 절대 커밋하지 마세요.
- [ ] 텍스트, 표, 이미지 추출 수를 확인하고 이미지 파일을 직접 검토했습니다.
- [ ] 텍스트 RAG와 멀티모달 답변을 같은 질문으로 비교하고 차이를 설명했습니다.
- [ ] 답변 5건의 근거를 원본 자료와 대조해 README에 기록했습니다.
- [ ] 노트북을 위에서 아래로 실행한 결과가 저장되어 있습니다.